In [2]:
pip install -U scikit-learn

   ---------------------------------------- 0.0/8.1 MB ? eta -:--:--
   ------------------ --------------------- 3.7/8.1 MB 18.9 MB/s eta 0:00:01
   ---------------------------------------- 8.1/8.1 MB 21.7 MB/s  0:00:00
   ---------------------------------------- 0.0/37.3 MB ? eta -:--:--
   --- ------------------------------------ 3.7/37.3 MB 30.1 MB/s eta 0:00:02
   -------- ------------------------------- 7.6/37.3 MB 21.1 MB/s eta 0:00:02
   ------------- -------------------------- 12.6/37.3 MB 22.0 MB/s eta 0:00:02
   --------------- ------------------------ 14.7/37.3 MB 20.2 MB/s eta 0:00:02
   -------------------- ------------------- 19.1/37.3 MB 19.5 MB/s eta 0:00:01
   -------------------------- ------------- 25.2/37.3 MB 21.1 MB/s eta 0:00:01
   --------------------------------- ------ 30.9/37.3 MB 22.1 MB/s eta 0:00:01
   ---------------------------------------  36.7/37.3 MB 22.8 MB/s eta 0:00:01
   ---------------------------------------  37.2/37.3 MB 22.9 MB/s eta 0:00:01
 


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\ASUS\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    roc_curve, precision_recall_curve, f1_score, recall_score, 
    precision_score, auc
)
import warnings
warnings.filterwarnings('ignore')

In [2]:
print("="*80)
print("STUDENT DROPOUT PREDICTION MODEL")
print("="*80)

STUDENT DROPOUT PREDICTION MODEL


In [3]:
df = pd.read_csv('student_data.csv', sep=';')
print(f"\n1. DATA LOADING")
print(f"   Dataset shape: {df.shape}")
print(f"   Columns: {df.shape[1]}")
print(f"   Target variable unique values: {df.iloc[:, -1].unique()}")
print(f"   Target variable value counts:\n{df.iloc[:, -1].value_counts()}")
target_col = df.columns[-1]
target_name = df[target_col].name
feature_cols = df.columns[:-1]
df_clean = df.dropna(subset=[target_col])
print(f"\n2. DATA CLEANING")
print(f"   Rows after removing target NaN: {len(df_clean)}")



1. DATA LOADING
   Dataset shape: (4424, 37)
   Columns: 37
   Target variable unique values: ['Dropout' 'Graduate' 'Enrolled']
   Target variable value counts:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

2. DATA CLEANING
   Rows after removing target NaN: 4424


In [4]:
target_col = df.columns[-1]
target_name = df[target_col].name
feature_cols = df.columns[:-1]
df_clean = df.dropna(subset=[target_col])
print(f"\n2. DATA CLEANING")
print(f"   Rows after removing target NaN: {len(df_clean)}")
numeric_cols = df_clean[feature_cols].select_dtypes(include=[np.number]).columns
categorical_cols = df_clean[feature_cols].select_dtypes(include=['object']).columns
print(f"   Numeric features: {len(numeric_cols)}")
print(f"   Categorical features: {len(categorical_cols)}")
for col in numeric_cols:
    missing_pct = df_clean[col].isna().sum() / len(df_clean) * 100
    if missing_pct > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f"   Filled {col} with median value: {median_val:.2f}")
df_processed = df_clean.copy()



2. DATA CLEANING
   Rows after removing target NaN: 4424
   Numeric features: 36
   Categorical features: 0


In [5]:
le_dict = {}
for col in categorical_cols:
    le = LabelEncoder()
    df_processed[col] = le.fit_transform(df_processed[col].astype(str))
    le_dict[col] = le
    print(f"   Encoded {col}: {len(le.classes_)} unique values")
X = df_processed[feature_cols]
y = df_processed[target_col]
y_binary = (y == 'Dropout').astype(int)
print(f"\n3. TARGET VARIABLE PREPARATION")
print(f"   Class distribution (binary):")
print(f"   - Dropout (1): {(y_binary==1).sum()} ({(y_binary==1).sum()/len(y_binary)*100:.1f}%)")
print(f"   - Not Dropout (0): {(y_binary==0).sum()} ({(y_binary==0).sum()/len(y_binary)*100:.1f}%)")
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"\n4. DATA SPLITTING & SCALING")
print(f"   Training set: {X_train.shape}")
print(f"   Test set: {X_test.shape}")
print(f"   Features scaled using StandardScaler")



3. TARGET VARIABLE PREPARATION
   Class distribution (binary):
   - Dropout (1): 1421 (32.1%)
   - Not Dropout (0): 3003 (67.9%)

4. DATA SPLITTING & SCALING
   Training set: (3539, 36)
   Test set: (885, 36)
   Features scaled using StandardScaler


In [7]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
}
print(f"\n5. MODEL TRAINING & EVALUATION")
print("="*80)
results = {}
predictions = {}
for name, model in models.items():
    print(f"\n{name}")
    print("-" * 40)
    
    if name == 'Logistic Regression':
        model.fit(X_train_scaled, y_train)
        y_pred_train = model.predict(X_train_scaled)
        y_pred = model.predict(X_test_scaled)
        y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred_train = model.predict(X_train)
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    train_f1 = f1_score(y_train, y_pred_train)
    test_f1 = f1_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'model': model,
        'train_f1': train_f1,
        'test_f1': test_f1,
        'precision': precision,
        'recall': recall,
        'roc_auc': roc_auc,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    predictions[name] = y_pred
    
    print(f"   Train F1-Score: {train_f1:.4f}")
    print(f"   Test F1-Score: {test_f1:.4f}")
    print(f"   Precision: {precision:.4f}")
    print(f"   Recall: {recall:.4f}")
    print(f"   ROC-AUC: {roc_auc:.4f}")
    
    print(f"\n   Classification Report:")
    print(classification_report(y_test, y_pred, target_names=['Not Dropout', 'Dropout']))
    
    cm = confusion_matrix(y_test, y_pred)
    print(f"   Confusion Matrix:")
    print(f"   TN={cm[0,0]}, FP={cm[0,1]}, FN={cm[1,0]}, TP={cm[1,1]}")
best_model_name = max(results.keys(), key=lambda x: results[x]['roc_auc'])
print(f"\n6. MODEL COMPARISON")
print("="*80)
print(f"{'Model':<25} {'Train F1':<12} {'Test F1':<12} {'Precision':<12} {'Recall':<12} {'ROC-AUC':<12}")
print("-" * 80)
for name, result in results.items():
    marker = " <- BEST" if name == best_model_name else ""
    print(f"{name:<25} {result['train_f1']:<12.4f} {result['test_f1']:<12.4f} {result['precision']:<12.4f} {result['recall']:<12.4f} {result['roc_auc']:<12.4f}{marker}")
best_model = results[best_model_name]['model']
print(f"\n7. FEATURE IMPORTANCE ANALYSIS")
print("="*80)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1]
    print(f"\nTop 15 Most Important Features ({best_model_name}):")
    print("-" * 50)
    for i in range(min(15, len(indices))):
        feat_idx = indices[i]
        feat_name = feature_cols[feat_idx]
        importance = importances[feat_idx]
        print(f"{i+1:2d}. {feat_name:<40} {importance:.4f}")
elif hasattr(best_model, 'coef_'):
    coef = np.abs(best_model.coef_[0])
    indices = np.argsort(coef)[::-1]
    print(f"\nTop 15 Most Important Features ({best_model_name}):")
    print("-" * 50)
    for i in range(min(15, len(indices))):
        feat_idx = indices[i]
        feat_name = feature_cols[feat_idx]
        importance = coef[feat_idx]
        print(f"{i+1:2d}. {feat_name:<40} {importance:.4f}")
print(f"\n8. PREDICTION EXAMPLE")
print("="*80)
sample_idx = 0
sample = X_test.iloc[sample_idx:sample_idx+1]
actual = y_test.iloc[sample_idx]
if best_model_name == 'Logistic Regression':
    sample_scaled = scaler.transform(sample)
    pred_proba = best_model.predict_proba(sample_scaled)[0, 1]
else:
    pred_proba = best_model.predict_proba(sample)[0, 1]
print(f"Sample Student Features (row 0 from test set):")
for feat, val in zip(feature_cols[:10], sample.iloc[0, :10].values):
    print(f"  {feat}: {val:.2f}")
print(f"\nActual Status: {'Dropout' if actual == 1 else 'Not Dropout'}")
print(f"Predicted Dropout Probability: {pred_proba:.4f}")
print(f"Risk Level: {'HIGH RISK' if pred_proba > 0.7 else 'MEDIUM RISK' if pred_proba > 0.4 else 'LOW RISK'}")
print(f"\n9. MODEL CROSS-VALIDATION")
print("="*80)
cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='roc_auc')
print(f"{best_model_name} ROC-AUC Cross-Validation Scores:")
print(f"  Fold 1: {cv_scores[0]:.4f}")
print(f"  Fold 2: {cv_scores[1]:.4f}")
print(f"  Fold 3: {cv_scores[2]:.4f}")
print(f"  Fold 4: {cv_scores[3]:.4f}")
print(f"  Fold 5: {cv_scores[4]:.4f}")
print(f"  Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"\n10. SUMMARY & RECOMMENDATIONS")
print("="*80)
print(f"Best Performing Model: {best_model_name}")
print(f"ROC-AUC Score: {results[best_model_name]['roc_auc']:.4f}")
print(f"Recall (Dropout Detection): {results[best_model_name]['recall']:.4f}")
print(f"\nKey Insights:")
print(f"  - The model successfully identifies {results[best_model_name]['recall']*100:.1f}% of at-risk students")
print(f"  - Precision of {results[best_model_name]['precision']*100:.1f}% means identified students are likely to dropout")
print(f"  - Imbalanced data handled through stratified split and appropriate metrics")
print(f"\nRecommendations for Early Intervention:")
print(f"  - Target students with predicted dropout probability > 0.6")
print(f"  - Focus on the top 5 predictive features for personalized support")
print(f"  - Monitor academic performance metrics closely in first semester")
print(f"  - Provide academic counseling and tutoring support proactively")
print(f"\n{'='*80}")


print(f"\n11. AT-RISK STUDENT LIST")
print("="*80)

all_proba = best_model.predict_proba(X)[:, 1]

def risk_level(p):
    if p >= 0.7:
        return 'HIGH RISK'
    elif p >= 0.4:
        return 'MEDIUM RISK'
    else:
        return 'LOW RISK'

at_risk_df = df_clean.copy().reset_index(drop=True)
at_risk_df['Student_ID']          = at_risk_df.index + 1
at_risk_df['Dropout_Probability']  = (all_proba * 100).round(2)
at_risk_df['Risk_Level']           = [risk_level(p) for p in all_proba]
at_risk_df['Actual_Status']        = at_risk_df[target_col]
at_risk_df['Predicted_Dropout']    = np.where(all_proba >= 0.5, 'Yes', 'No')

key_features = [
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Admission grade',
    'Age at enrollment',
    'Tuition fees up to date',
    'Debtor',
    'Scholarship holder',
    'Gender',
]
key_features_present = [f for f in key_features if f in at_risk_df.columns]

output_cols = ['Student_ID', 'Dropout_Probability', 'Risk_Level',
               'Actual_Status', 'Predicted_Dropout'] + key_features_present

at_risk_sorted = (
    at_risk_df[output_cols]
    .sort_values('Dropout_Probability', ascending=False)
    .reset_index(drop=True)
)

high_risk   = at_risk_sorted[at_risk_sorted['Risk_Level'] == 'HIGH RISK']
medium_risk = at_risk_sorted[at_risk_sorted['Risk_Level'] == 'MEDIUM RISK']
low_risk    = at_risk_sorted[at_risk_sorted['Risk_Level'] == 'LOW RISK']

print(f"\n  Risk Level Breakdown:")
print(f"  - HIGH RISK   (>= 70% dropout probability): {len(high_risk):>4} students")
print(f"  - MEDIUM RISK (40-69% dropout probability): {len(medium_risk):>4} students")
print(f"  - LOW RISK    (< 40% dropout probability):  {len(low_risk):>4} students")

print(f"\n  Top 20 Highest-Risk Students:")
print(f"  {'ID':>6}  {'Dropout%':>9}  {'Risk Level':<13}  {'Actual Status':<10}  "
      f"{'2nd Sem Approved':>16}  {'2nd Sem Grade':>13}  {'Tuition OK':>10}  {'Debtor':>6}")
print("  " + "-"*100)
for _, row in at_risk_sorted.head(20).iterrows():
    approved = row.get('Curricular units 2nd sem (approved)', 'N/A')
    grade    = row.get('Curricular units 2nd sem (grade)', 'N/A')
    tuition  = 'Yes' if row.get('Tuition fees up to date', 0) == 1 else 'No'
    debtor   = 'Yes' if row.get('Debtor', 0) == 1 else 'No'
    print(f"  {int(row['Student_ID']):>6}  {row['Dropout_Probability']:>8.1f}%  "
          f"{row['Risk_Level']:<13}  {row['Actual_Status']:<10}  "
          f"{str(approved):>16}  {str(grade):>13}  {tuition:>10}  {debtor:>6}")

at_risk_sorted.to_csv('at_risk_students.csv', index=False)
print(f"\n  Full list saved to: at_risk_students.csv")
print(f"  Total students assessed: {len(at_risk_sorted)}")
print(f"  Students needing attention (HIGH + MEDIUM): {len(high_risk) + len(medium_risk)}")

print(f"\n{'='*80}")
print("MODEL TRAINING COMPLETED SUCCESSFULLY")
print(f"{'='*80}\n")
   


5. MODEL TRAINING & EVALUATION

Logistic Regression
----------------------------------------
   Train F1-Score: 0.7914
   Test F1-Score: 0.8054
   Precision: 0.8894
   Recall: 0.7359
   ROC-AUC: 0.9266

   Classification Report:
              precision    recall  f1-score   support

 Not Dropout       0.88      0.96      0.92       601
     Dropout       0.89      0.74      0.81       284

    accuracy                           0.89       885
   macro avg       0.89      0.85      0.86       885
weighted avg       0.89      0.89      0.88       885

   Confusion Matrix:
   TN=575, FP=26, FN=75, TP=209

Decision Tree
----------------------------------------
   Train F1-Score: 0.9030
   Test F1-Score: 0.7193
   Precision: 0.7510
   Recall: 0.6901
   ROC-AUC: 0.8136

   Classification Report:
              precision    recall  f1-score   support

 Not Dropout       0.86      0.89      0.88       601
     Dropout       0.75      0.69      0.72       284

    accuracy                      